# Build a MovieLens recommendation baseline with Nori

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Synthefy/synthefy-nori/blob/codex/movielens-recommendation-notebook/examples/notebooks/nori-movielens-recommendations.ipynb)

Use Nori-6M to predict movie ratings, compare it with movie-average and SVD++ baselines, and rank ten unseen movies for one user. This notebook accompanies [Build a MovieLens Recommendation Baseline with Nori in Minutes](https://synthefy.com/blog/nori-recommendation-systems).

Use Python 3.11 or 3.12 and run the cells in order. The full 80,000-context/20,000-query run needs a GPU with at least 40 GB of memory, such as an A100 or H100 (**Colab: Runtime → Change runtime type**). On a smaller GPU or CPU, set `SMOKE_TEST = True` to check the code with 512 context rows and 32 test rows; this subset cannot reproduce the blog results. The notebook downloads public data and weights and needs no API key.

**Saved outputs:** every code cell includes results from the full split, run on October 4, 2026, using an NVIDIA H100, a 40% per-process GPU memory limit, and four CPU threads. The configuration cell prints package versions.

## 1. Install the public packages

These SDK and runtime versions were the latest stable releases checked on October 4, 2026. Install before importing packages; restart the kernel if you already imported other versions. The configuration cell prints the installed versions.

In [1]:
%pip install -q "synthefy==7.1.3" "synthefy-nori==0.21.0" "scikit-surprise==1.1.5" "huggingface-hub>=1,<2"

Note: you may need to restart the kernel to use updated packages.

In [2]:
import hashlib
import io
import json
import os
import urllib.request
import zipfile
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from surprise import Dataset, Reader, SVDpp
from synthefy_nori import MemoryPolicy, NoriRegressor

DATA_DIR = Path("data/ml-100k")
DEVICE = os.environ.get("MOVIELENS_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
# Set True for a small end-to-end check. These subset scores are NOT the blog benchmark.
SMOKE_TEST = False
SEED = 0
SHRINK = 5.0
GENRES = [
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
    "Romance", "Sci-Fi", "Thriller", "War", "Western",
]
PACKAGE_VERSIONS = {
    name: version(name)
    for name in ("synthefy", "synthefy-nori", "numpy", "pandas", "torch", "scikit-surprise")
}
print(json.dumps({"packages": PACKAGE_VERSIONS, "device": DEVICE, "smoke_test": SMOKE_TEST}, indent=2))

{
  "packages": {
    "synthefy": "7.1.3",
    "synthefy-nori": "0.21.0",
    "numpy": "2.5.3",
    "pandas": "3.0.6",
    "torch": "2.10.0",
    "scikit-surprise": "1.1.5"
  },
  "device": "cuda:0",
  "smoke_test": false
}


## 2. Download MovieLens 100K and inspect the tables

[MovieLens 100K](https://grouplens.org/datasets/movielens/100k/) contains 100,000 ratings from 943 users for 1,682 movies. Its `u1.base` and `u1.test` files hold 80,000 and 20,000 ratings. This split holds out ratings without enforcing chronological order.

The code extracts four files and checks their SHA-256 hashes. See GroupLens for [usage terms](https://grouplens.org/datasets/movielens/100k/) and Harper and Konstan's [*The MovieLens Datasets: History and Context*](https://doi.org/10.1145/2827872) (2015) for the dataset's background.

In [3]:
DATA_SHA256 = {
    "u1.base": "ce253ec86c448b44fb3ba9a30d12dcfc2e9210cbde71efada3730c22e9ac212a",
    "u1.test": "18c6014a4b2c7324f250a63f8904a7b16b2b19f911129e346141507b0cbac950",
    "u.user": "f120e114da2e8cf314fd28f99417c94ae9ddf1cb6db8ce0e4b5995d40e90e62c",
    "u.item": "553841ebc7de3a0fd0d6b62a204ea30c1e651aacfb2814c7a6584ac52f2c5701",
}


def download_movielens_files(data_dir):
    """Download the four benchmark files and verify their contents."""
    data_dir.mkdir(parents=True, exist_ok=True)
    if not all((data_dir / name).exists() for name in DATA_SHA256):
        url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
        with urllib.request.urlopen(url, timeout=120) as response:
            archive_bytes = response.read()
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
            for name, expected in DATA_SHA256.items():
                payload = archive.read(f"ml-100k/{name}")
                if hashlib.sha256(payload).hexdigest() != expected:
                    raise ValueError(f"Unexpected dataset contents: {name}")
                (data_dir / name).write_bytes(payload)
    for name, expected in DATA_SHA256.items():
        if hashlib.sha256((data_dir / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"Cached {name} does not match MovieLens 100K; remove it and rerun.")


def read_movielens_ratings(name):
    """Load one supplied rating split, preserving its row order."""
    return pd.read_csv(
        DATA_DIR / name, sep="\t", names=["user_id", "item_id", "rating", "timestamp"]
    )


download_movielens_files(DATA_DIR)
train_ratings_table = read_movielens_ratings("u1.base")
test_ratings_table = read_movielens_ratings("u1.test")
raw_users = pd.read_csv(
    DATA_DIR / "u.user", sep="|", names=["user_id", "age", "gender", "occupation", "zip"]
)
raw_items = pd.read_csv(DATA_DIR / "u.item", sep="|", encoding="latin-1", header=None)
movie_titles = raw_items[[0, 1]].set_axis(["item_id", "title"], axis=1)

assert len(train_ratings_table) == 80_000 and len(test_ratings_table) == 20_000
assert len(raw_users) == 943 and len(raw_items) == 1_682
if SMOKE_TEST:
    train_ratings_table = train_ratings_table.sample(n=512, random_state=SEED).reset_index(drop=True)
    test_ratings_table = test_ratings_table.sample(n=32, random_state=SEED).reset_index(drop=True)
    print("SMOKE TEST: subset only; do not compare these scores with the blog benchmark.")

users = raw_users.drop(columns="zip").copy()
for column in ("gender", "occupation"):
    categories = pd.Categorical(users[column])
    print(f"{column} codes:", dict(enumerate(categories.categories)))
    users[column] = categories.codes
users["age_band"] = users["age"] // 10
items = raw_items[[0, 2, *range(5, 24)]].set_axis(["item_id", "release_date", *GENRES], axis=1)
items["year"] = pd.to_datetime(items.pop("release_date"), format="%d-%b-%Y").dt.year

print(f"Context ratings: {len(train_ratings_table):,}; held-out ratings: {len(test_ratings_table):,}")
display(train_ratings_table.head())
display(raw_users.drop(columns="zip").head())
display(movie_titles.merge(items[["item_id", "year", "Action", "Animation", "Comedy"]], on="item_id").head())

gender codes: {0: 'F', 1: 'M'}
occupation codes: {0: 'administrator', 1: 'artist', 2: 'doctor', 3: 'educator', 4: 'engineer', 5: 'entertainment', 6: 'executive', 7: 'healthcare', 8: 'homemaker', 9: 'lawyer', 10: 'librarian', 11: 'marketing', 12: 'none', 13: 'other', 14: 'programmer', 15: 'retired', 16: 'salesman', 17: 'scientist', 18: 'student', 19: 'technician', 20: 'writer'}
Context ratings: 80,000; held-out ratings: 20,000


,user_id,item_id,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,3,4,878542960
3,1,4,3,876893119
4,1,5,3,889751712


,user_id,age,gender,occupation
0,1,24,M,technician
1,2,53,F,other
2,3,23,M,writer
3,4,24,M,technician
4,5,33,F,other


,item_id,title,year,Action,Animation,Comedy
0,1,Toy Story (1995),1995.0,0,1,1
1,2,GoldenEye (1995),1995.0,1,0,0
2,3,Four Rooms (1995),1995.0,0,0,0
3,4,Get Shorty (1995),1995.0,1,0,1
4,5,Copycat (1995),1995.0,0,0,0


## 3. Build the 37 input features

Each user–movie row contains IDs, age, gender and occupation codes, age band, 19 genre indicators, release year, and rating-history statistics. Titles are used only for display.

History features include movie and user means, movie means within demographic groups (gender, age band, occupation), the count behind each mean, and genre affinity. Means are smoothed toward the available history's global mean with a pseudo-count of five. Genre affinity pools the user's ratings across the target movie's genres; a past movie contributes once per matching genre.

**Keep each training row's label out of its own features.** Split training rows into five folds with `default_rng(1)` and compute each row's statistics from the other four. Held-out and candidate rows use the full training history. Exclude `rating` and `timestamp` from the feature matrix; leave missing release years for Nori to handle.

In [4]:
def join_pair_metadata(pairs):
    """Attach user and movie attributes while preserving pair order."""
    return (
        pairs.merge(users, on="user_id", how="left", validate="many_to_one")
        .merge(items, on="item_id", how="left", validate="many_to_one")
    )


def compute_group_rating_means(history, pairs, keys):
    """Look up smoothed rating means and counts for each pair's group."""
    stats = history.groupby(keys)["rating"].agg(["sum", "count"])
    joined = pairs[keys].join(stats, on=keys).fillna(0)
    mean = (joined["sum"] + SHRINK * history["rating"].mean()) / (joined["count"] + SHRINK)
    return pd.DataFrame({"mean": mean, "n": joined["count"]})


def compute_user_genre_affinity(history, pairs):
    """Pool each user's historical ratings over the target movie's genres."""
    genre_sum = (history[GENRES] * history["rating"].to_numpy()[:, None]).groupby(history["user_id"]).sum()
    genre_count = history.groupby("user_id")[GENRES].sum()
    flags = pairs[GENRES].to_numpy()
    total = np.nan_to_num(genre_sum.reindex(pairs["user_id"]).to_numpy() * flags).sum(axis=1)
    count = np.nan_to_num(genre_count.reindex(pairs["user_id"]).to_numpy() * flags).sum(axis=1)
    return pd.Series((total + SHRINK * history["rating"].mean()) / (count + SHRINK), index=pairs.index)


def build_pair_features(history, pairs):
    """Build numeric pair features from a separate labeled history."""
    features = pairs.drop(columns=["rating", "timestamp"], errors="ignore").copy()
    groups = [
        ("item_mean", ["item_id"]),
        ("item_mean_by_gender", ["item_id", "gender"]),
        ("item_mean_by_age_band", ["item_id", "age_band"]),
        ("item_mean_by_occupation", ["item_id", "occupation"]),
        ("user_mean", ["user_id"]),
    ]
    for name, keys in groups:
        stats = compute_group_rating_means(history, pairs, keys)
        features[name] = stats["mean"]
        features[f"{name}_n"] = stats["n"]
    features["genre_affinity"] = compute_user_genre_affinity(history, pairs)
    return features


def build_context_features(history):
    """Create five-fold history features without each row's own label."""
    folds = np.random.default_rng(1).integers(0, 5, len(history))
    return pd.concat([
        build_pair_features(history.loc[folds != fold], history.loc[folds == fold])
        for fold in range(5)
    ]).loc[history.index]


history = join_pair_metadata(train_ratings_table)
# Only pair IDs are supplied when constructing query features.
test_pairs = join_pair_metadata(test_ratings_table[["user_id", "item_id"]])
train_features = build_context_features(history)
test_features = build_pair_features(history, test_pairs)
train_ratings = train_ratings_table["rating"]
test_ratings = test_ratings_table["rating"]
feature_columns = train_features.columns.tolist()

assert len(feature_columns) == 37
assert test_features.columns.tolist() == feature_columns
assert not {"rating", "timestamp", "title"}.intersection(feature_columns)
print("Input columns:", feature_columns)
shown = ["user_id", "item_id", "age", "gender", "occupation", "year", "Action", "Animation", "Comedy",
         "item_mean", "user_mean", "genre_affinity"]
print("Context examples (12 of 37 input columns, plus the separate target):")
display(train_features[shown].head().assign(rating=train_ratings.head()))
print("Query examples (ratings withheld):")
display(test_features[shown].head())

Input columns: ['user_id', 'item_id', 'age', 'gender', 'occupation', 'age_band', 'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'year', 'item_mean', 'item_mean_n', 'item_mean_by_gender', 'item_mean_by_gender_n', 'item_mean_by_age_band', 'item_mean_by_age_band_n', 'item_mean_by_occupation', 'item_mean_by_occupation_n', 'user_mean', 'user_mean_n', 'genre_affinity']
Context examples (12 of 37 input columns, plus the separate target):


,user_id,item_id,age,gender,occupation,year,Action,Animation,Comedy,item_mean,user_mean,genre_affinity,rating
0,1,1,24,1,19,1995.0,0,1,1,3.862243,3.620570,3.060702,5
1,1,2,24,1,19,1995.0,1,0,0,3.260255,3.620570,3.084403,3
2,1,3,24,1,19,1995.0,0,0,0,3.056897,3.622969,3.506148,4
3,1,4,24,1,19,1995.0,1,0,1,3.526411,3.739859,3.742224,3
4,1,5,24,1,19,1995.0,0,0,0,3.277447,3.632600,3.773417,3


Query examples (ratings withheld):


,user_id,item_id,age,gender,occupation,year,Action,Animation,Comedy,item_mean,user_mean,genre_affinity
0,1,6,24,1,19,1995.0,0,0,0,3.425670,3.676013,3.881319
1,1,10,24,1,19,1996.0,0,0,0,3.854381,3.676013,3.832997
2,1,12,24,1,19,1995.0,0,0,0,4.377971,3.676013,3.591044
3,1,14,24,1,19,1994.0,0,0,0,3.887184,3.676013,3.960021
4,1,17,24,1,19,1996.0,1,0,1,3.203140,3.676013,3.432595


## 4. Predict ratings with Nori

`fit` supplies the context; it does not update pretrained weights. `allow_subsample=False` retains the full context. The runtime reports which memory path it used. Clip predictions to the one-to-five rating scale for RMSE; leave scores unrounded for ranking.

In [5]:
np.random.seed(SEED)
torch.manual_seed(SEED)
rating_model = NoriRegressor(
    model="nori-6m",
    device=DEVICE,
    encode_id_cols=["user_id", "item_id"],
    memory_policy=MemoryPolicy(cache=True, allow_subsample=False),
)
rating_model.fit(train_features, train_ratings)
predicted_ratings = np.clip(rating_model.predict(test_features), 1, 5)
assert np.isfinite(predicted_ratings).all()
assert rating_model.memory_report_["dropped_context_rows"] == 0
print("Memory path:", rating_model.memory_report_["rung"])
display(test_ratings_table[["user_id", "item_id", "rating"]].head().assign(prediction=predicted_ratings[:5]))

/tmp/nori-blog-sdk-check/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Memory path: no_cache


,user_id,item_id,rating,prediction
0,1,6,5,3.597584
1,1,10,3,4.191908
2,1,12,5,4.464228
3,1,14,5,4.303072
4,1,17,3,3.166684


## 5. Measure rating accuracy against the baselines

**SVD++ is an established collaborative-filtering baseline**, introduced by Yehuda Koren of AT&T Labs Research in [*Factorization Meets the Neighborhood* (KDD 2008)](https://www.cs.cornell.edu/courses/cs6241/2020sp/readings/Koren-2008-Factorization.pdf). It learns user and movie representations from ratings and from which movies each user has rated. This gives us a model trained for rating prediction to compare with Nori's fixed weights.

We use Nicolas Hug's [Surprise library](https://joss.theoj.org/papers/10.21105/joss.02174), with [default SVD++ settings](https://surprise.readthedocs.io/en/stable/matrix_factorization.html#surprise.prediction_algorithms.matrix_factorization.SVDpp), `random_state=0`, and no tuning. SVD++ has the lowest RMSE in Surprise's [published MovieLens 100K benchmark](https://github.com/NicolasHug/Surprise#benchmarks). That benchmark averages five folds; we measure the supplied `u1.base` / `u1.test` split.

Root mean squared error (RMSE) measures rating error; lower is better. The movie-average baseline uses the smoothed `item_mean`. SVD++ uses IDs and ratings; Nori uses all 37 features, so the approaches receive different inputs.

The next cell computes the results. Package versions, hardware, and memory execution paths may affect repeat runs. This single split establishes neither statistical significance nor state-of-the-art performance.

The original experiment adopted these five-fold history features after inspecting a test result, so its results are exploratory. Rating RMSE does not establish ranking quality or accuracy for users with no history.

In [6]:
def compute_rating_rmse(actual, predicted):
    """Measure squared rating error after clipping to the rating scale."""
    return float(np.sqrt(np.mean((np.asarray(actual) - np.clip(predicted, 1, 5)) ** 2)))


surprise_data = Dataset.load_from_df(
    train_ratings_table[["user_id", "item_id", "rating"]], Reader(rating_scale=(1, 5))
)
svd_model = SVDpp(random_state=SEED).fit(surprise_data.build_full_trainset())
svd_predictions = np.array([
    svd_model.predict(user_id, item_id).est
    for user_id, item_id in test_ratings_table[["user_id", "item_id"]].itertuples(index=False, name=None)
])
results = pd.DataFrame([
    {"method": "Movie average", "RMSE": compute_rating_rmse(test_ratings, test_features["item_mean"])},
    {"method": "SVD++ (Surprise defaults)", "RMSE": compute_rating_rmse(test_ratings, svd_predictions)},
    {"method": "Nori-6M", "RMSE": compute_rating_rmse(test_ratings, predicted_ratings)},
])
print("SMOKE TEST subset results" if SMOKE_TEST else "MovieLens 100K: u1.base → u1.test")
display(results.round({"RMSE": 4}))

MovieLens 100K: u1.base → u1.test


,method,RMSE
0,Movie average,1.0370
1,SVD++ (Surprise defaults),0.9330
2,Nori-6M,0.9289


## 6. Recommend ten unseen movies to one user

For an existing user, score every catalog movie absent from their context history. Build candidate features from the full context history, without inspecting held-out ratings. Break score ties by movie ID and add titles after inference.

The list below has not been evaluated for ranking quality. To evaluate it, fix the candidate set, relevance rule, and K, then measure Recall@K or NDCG@K. Test users with no history separately.

In [7]:
example_user_id = int(train_ratings_table["user_id"].iloc[0])
seen_item_ids = train_ratings_table.loc[
    train_ratings_table["user_id"] == example_user_id, "item_id"
]
unseen_item_ids = items.loc[~items["item_id"].isin(seen_item_ids), "item_id"]
if SMOKE_TEST:
    unseen_item_ids = unseen_item_ids.head(20)
candidate_pairs = pd.DataFrame({"user_id": example_user_id, "item_id": unseen_item_ids.to_numpy()})
candidate_features = build_pair_features(history, join_pair_metadata(candidate_pairs))[feature_columns]
candidates = candidate_pairs.assign(score=rating_model.predict(candidate_features))
assert np.isfinite(candidates["score"]).all()
assert not candidates["item_id"].isin(seen_item_ids).any()
assert not candidates.duplicated(["user_id", "item_id"]).any()
top_10 = (
    candidates.sort_values(["score", "item_id"], ascending=[False, True])
    .head(10)
    .merge(movie_titles, on="item_id", how="left", validate="one_to_one")
    [["user_id", "item_id", "title", "score"]]
)
display(top_10)

,user_id,item_id,title,score
0,1,64,"Shawshank Redemption, The (1994)",4.609333
1,1,318,Schindler's List (1993),4.582310
2,1,483,Casablanca (1942),4.564101
3,1,1142,When We Were Kings (1996),4.518218
4,1,357,One Flew Over the Cuckoo's Nest (1975),4.515006
5,1,484,"Maltese Falcon, The (1941)",4.496486
6,1,408,"Close Shave, A (1995)",4.495269
7,1,98,"Silence of the Lambs, The (1991)",4.488810
8,1,313,Titanic (1997),4.488481
9,1,603,Rear Window (1954),4.488107


## Use your own ratings and metadata

Replace the input tables and choose a split that matches when predictions will be made. For time-dependent data, compute history features using only information available before each prediction.

The [Nori quickstart](https://docs.synthefy.com/nori/quickstart) documents the prediction interface.